# 03 - Uji prediksi + keputusan

Robot **belum** dijalankan. Hanya cek deteksi dan logika keputusan.

In [ ]:
%load_ext autoreload
%autoreload 2

import cv2
import matplotlib.pyplot as plt

from robot.config import Config
from robot.vision import Detector
from robot.decision import decide

cfg = Config()
cfg.imgsz = 640          # untuk uji diam, akurasi lebih penting
det = Detector(cfg.model, cfg.camera, cfg.imgsz, cfg.conf, cfg.target_class)
print("Kelas model:", det.names)

In [ ]:
frame = det.read()
out = det.infer(frame)
print("deteksi  :", out["det"])
print("keputusan:", decide(out["det"], out["frame_wh"][0], cfg))

plt.figure(figsize=(9, 6)); plt.axis("off")
plt.imshow(cv2.cvtColor(out["plot"], cv2.COLOR_BGR2RGB)); plt.show()

## Preview live (tanpa memblok kernel)

In [ ]:
import asyncio
import ipywidgets as W
from IPython.display import display

img = W.Image(format="jpeg")
display(img)

async def preview():
    while True:
        f = det.read()
        if f is not None:
            o = det.infer(f)
            ok, b = cv2.imencode(".jpg", o["plot"], [cv2.IMWRITE_JPEG_QUALITY, 70])
            if ok:
                img.value = b.tobytes()
        await asyncio.sleep(0.01)

preview_task = asyncio.create_task(preview())

In [ ]:
preview_task.cancel()
det.close()